# Fantasy Football Analytics — RB Dataset Builder (2015–2025)

**Data sources**
- `nfl_data_py` — seasonal stats, rosters, PBP, snap counts, depth charts, injuries, NGS rushing
- **Sleeper API** — trending waiver-wire adds/drops (real-time demo; IDs come from `nfl_data_py` rosters)

**Output:** `rb_features_2015_2025.csv` / `.parquet` — 61 features ready for XGBoost

Run cells top-to-bottom, or jump to **Step-by-Step Build** to execute section by section.


## Imports

In [ ]:
from __future__ import annotations

import argparse
import json
import logging
import time
from pathlib import Path
from typing import Optional

import nfl_data_py as nfl
import numpy as np
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from urllib3.util.retry import Retry

## Config

Adjust `START_YEAR`, `END_YEAR`, or `MIN_GAMES` here before running.
Set `SKIP_PBP = False` to include goal-line and `carries_share` features (slow — ~20 min).


In [ ]:
START_YEAR     = 2015
END_YEAR       = 2025
MIN_GAMES      = 4        # drop injury-truncated seasons (noise floor)
GL_YARD_LINE   = 20       # 'goal line' = within opponent's 20-yard line
PBP_CHUNK_SIZE = 4        # years of PBP in memory at once
SKIP_PBP       = True     # set False to include goal-line + carries_share (slow)

CACHE_DIR   = Path('.cache')
OUT_PARQUET = f'rb_features_{START_YEAR}_{END_YEAR}.parquet'
OUT_CSV     = f'rb_features_{START_YEAR}_{END_YEAR}.csv'

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)-8s  %(message)s',
    datefmt='%H:%M:%S',
)
log = logging.getLogger(__name__)
CACHE_DIR.mkdir(exist_ok=True)

## Sleeper API Client

No API key required. Responses cached to `.cache/` to avoid redundant network calls.
`fetch_trending()` returns real-time waiver-wire adds/drops — a live API integration demo.

> **Note on IDs:** `sleeper_id` values come from `nfl_data_py` rosters (99.4% coverage for 2015–2024),
> not from the Sleeper player endpoint, which only has `gsis_id` populated for ~32% of its player records.


In [ ]:
class SleeperClient:
    """
    Thin HTTP client for the Sleeper public API (no auth required).
    Disk-cached to avoid redundant network calls across runs.

    Note on IDs: sleeper_id values in the output dataset come from nfl_data_py's
    import_ids() crosswalk, which has better historical coverage than Sleeper's
    own /players/nfl endpoint (which only stores gsis_id for ~32% of players).
    This client is used here for real-time trending data.
    """

    BASE_URL = "https://api.sleeper.app/v1"

    def __init__(self) -> None:
        self._session = self._build_session()

    @staticmethod
    def _build_session() -> requests.Session:
        retry = Retry(
            total=4,
            backoff_factor=0.6,
            status_forcelist={429, 500, 502, 503, 504},
            allowed_methods={"GET"},
        )
        s = requests.Session()
        s.mount("https://", HTTPAdapter(max_retries=retry))
        s.headers.update({"Accept": "application/json", "User-Agent": "ff-analytics/1.0"})
        return s

    def _get(self, path: str, *, ttl_hours: int = 24) -> dict | list:
        cache_file = CACHE_DIR / (path.lstrip("/").replace("/", "_") + ".json")
        if cache_file.exists():
            if (time.time() - cache_file.stat().st_mtime) / 3600 < ttl_hours:
                return json.loads(cache_file.read_text(encoding="utf-8"))
        log.info("Sleeper GET %s", path)
        resp = self._session.get(f"{self.BASE_URL}{path}", timeout=30)
        resp.raise_for_status()
        data = resp.json()
        cache_file.write_text(json.dumps(data), encoding="utf-8")
        return data

    def fetch_trending(self, trend_type: str = "add", limit: int = 25) -> pd.DataFrame:
        """
        Trending waiver-wire adds or drops over the past 48 hours.
        trend_type: 'add' | 'drop'
        Returns player_id (Sleeper) + count columns.
        """
        data = self._get(f"/players/nfl/trending/{trend_type}", ttl_hours=1)
        df = pd.DataFrame(data)
        df["trend_type"] = trend_type
        log.info("Sleeper trending (%s): %d players", trend_type, len(df))
        return df

## Data Loaders (nfl_data_py)

### ID Crosswalk
Provides `gsis_id ↔ pfr_id` mapping needed by `load_snap_counts()`.


In [ ]:
def load_id_crosswalk() -> pd.DataFrame:
    """
    import_ids() provides gsis_id <-> pfr_id <-> sleeper_id mapping.
    Deduplicated to one row per gsis_id (latest db_season wins).
    Only rows with a valid GSIS ID (format 00-0######) are kept.
    """
    log.info("Loading ID crosswalk")
    ids = nfl.import_ids()

    ids = ids[ids["gsis_id"].str.match(r"00-\d{7}", na=False)].copy()
    ids = (
        ids.sort_values("db_season", ascending=False)
        .drop_duplicates(subset="gsis_id", keep="first")
    )

    keep = ["gsis_id", "pfr_id", "sleeper_id", "birthdate", "merge_name"]
    ids = ids[[c for c in keep if c in ids.columns]].copy()
    ids["sleeper_id"] = ids["sleeper_id"].apply(
        lambda x: str(int(x)) if pd.notna(x) else np.nan
    )
    log.info("ID crosswalk: %d valid GSIS entries", len(ids))
    return ids

### Seasonal Stats + Rosters
Base stats (carries, targets, EPA, TDs, etc.) merged with demographics (height, weight, draft number, age, college).


In [ ]:
def load_seasonal_rb(years: list[int]) -> pd.DataFrame:
    """
    Seasonal stats merged with roster demographics for RBs.
    One row per (player_id, season).
    """
    log.info("Loading seasonal stats %d-%d", years[0], years[-1])
    seasonal_frames: list[pd.DataFrame] = []
    for yr in years:
        try:
            seasonal_frames.append(nfl.import_seasonal_data(years=[yr], s_type="REG"))
        except Exception as exc:
            log.warning("Seasonal data unavailable for %d: %s", yr, exc)
    if not seasonal_frames:
        raise RuntimeError("No seasonal data loaded for any year in range")
    seasonal = pd.concat(seasonal_frames, ignore_index=True)

    log.info("Loading seasonal rosters %d-%d", years[0], years[-1])
    roster_frames: list[pd.DataFrame] = []
    for yr in years:
        try:
            r = nfl.import_seasonal_rosters(years=[yr])
            if "season" not in r.columns:
                r["season"] = yr
            roster_frames.append(r)
        except Exception as exc:
            log.warning("Roster unavailable for %d: %s", yr, exc)

    rosters = (
        pd.concat(roster_frames, ignore_index=True)
        .drop_duplicates(subset=["player_id", "season"], keep="last")
    )

    df = pd.merge(seasonal, rosters, on=["player_id", "season"], how="inner")
    df = df[df["position"] == "RB"].copy()
    df["ppg"] = df["fantasy_points_ppr"] / df["games"]

    log.info("Seasonal: %d RB-season rows across %d seasons", len(df), df["season"].nunique())
    return df

### PBP — Goal-Line Touches + Team Carry Totals
One chunked PBP load produces both tables in a single pass to minimise peak memory.
Controlled by `SKIP_PBP` in Config.

⚠️ **Slow** — allow 20–40 min for 2015–2025.


In [ ]:
def load_pbp_features(
    years: list[int], yard_line: int = GL_YARD_LINE
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Single chunked PBP load that produces two tables:

    goal_line : (player_id, season) -> gl_carries, gl_targets, gl_touches
                rushes + targets with yardline_100 <= yard_line

    team_carries : (team, season) -> team_carries_seas
                   total regular-season rush attempts per team
                   (denominator for carries_share)
    """
    log.info(
        "Loading PBP for goal-line + team carries %d-%d  (chunked, slow step)",
        years[0], years[-1],
    )

    gl_rush, gl_rec, tc_frames = [], [], []

    for i in range(0, len(years), PBP_CHUNK_SIZE):
        chunk = years[i : i + PBP_CHUNK_SIZE]
        log.info("  PBP chunk %d-%d", chunk[0], chunk[-1])

        pbp = nfl.import_pbp_data(years=chunk, downcast=True, cache=True)
        reg = pbp[pbp["season_type"] == "REG"]

        # --- team carry totals (all rush attempts, all positions)
        tc_frames.append(
            reg[reg["rush_attempt"] == 1]
            .groupby(["season", "posteam"])
            .size()
            .reset_index(name="team_carries_seas")
            .rename(columns={"posteam": "team"})
        )

        # --- goal-line rushes
        gl = reg[reg["yardline_100"] <= yard_line]
        gl_rush.append(
            gl[gl["rush_attempt"] == 1]
            .groupby(["season", "rusher_id"])
            .size()
            .reset_index(name="gl_carries")
            .rename(columns={"rusher_id": "player_id"})
        )

        # --- goal-line targets (pass_attempt covers completions + incompletions)
        gl_rec.append(
            gl[(gl["pass_attempt"] == 1) & gl["receiver_id"].notna()]
            .groupby(["season", "receiver_id"])
            .size()
            .reset_index(name="gl_targets")
            .rename(columns={"receiver_id": "player_id"})
        )

        del pbp, reg, gl

    goal_line = pd.merge(
        pd.concat(gl_rush, ignore_index=True),
        pd.concat(gl_rec,  ignore_index=True),
        on=["season", "player_id"],
        how="outer",
    ).fillna(0)
    goal_line["gl_carries"] = goal_line["gl_carries"].astype(int)
    goal_line["gl_targets"] = goal_line["gl_targets"].astype(int)
    goal_line["gl_touches"] = goal_line["gl_carries"] + goal_line["gl_targets"]

    team_carries = pd.concat(tc_frames, ignore_index=True)

    log.info(
        "PBP done: %d goal-line rows, %d team-season rows",
        len(goal_line), len(team_carries),
    )
    return goal_line, team_carries

### Snap Counts
Offensive snap % and snaps-per-game aggregated to season level, joined to GSIS IDs via the `pfr_id` crosswalk.


In [ ]:
def load_snap_counts(years: list[int], id_crosswalk: pd.DataFrame) -> pd.DataFrame:
    """
    Offensive snap percentage and snaps per game, aggregated to season level.
    Joined to GSIS IDs via the pfr_id column in the crosswalk.
    Returns one row per (player_id, season).
    """
    log.info("Loading snap counts %d-%d", years[0], years[-1])
    snaps = nfl.import_snap_counts(years=years)
    snaps = snaps[
        (snaps["position"] == "RB") & (snaps["game_type"] == "REG")
    ].copy()

    pfr_to_gsis = (
        id_crosswalk[["gsis_id", "pfr_id"]]
        .dropna(subset=["pfr_id"])
        .drop_duplicates(subset="pfr_id")
        .set_index("pfr_id")["gsis_id"]
        .to_dict()
    )
    snaps["player_id"] = snaps["pfr_player_id"].map(pfr_to_gsis)

    seasonal = (
        snaps[snaps["player_id"].notna()]
        .groupby(["player_id", "season"])
        .agg(
            avg_snap_pct=("offense_pct",   "mean"),
            off_snaps_pg=("offense_snaps", "mean"),
        )
        .reset_index()
    )
    log.info("Snap counts: %d player-season rows (%.0f%% matched to GSIS)",
             len(seasonal),
             snaps["player_id"].notna().mean() * 100)
    return seasonal

### Depth Charts
Week-1 RB depth chart position per player per season.
`depth_team_wk1 = 1` → starter, `2` → backup, `NaN` → not on roster / IR.


In [ ]:
def load_depth_charts(years: list[int]) -> pd.DataFrame:
    """
    Week-1 RB depth chart position per player per season.
    depth_team_wk1 = 1 (starter), 2 (backup), 3+ (depth).
    NaN means the player was not on the week-1 depth chart (injured/IR).
    """
    log.info("Loading depth charts %d-%d", years[0], years[-1])
    dc = nfl.import_depth_charts(years=years)

    dc_rb = dc[
        (dc["depth_position"] == "RB") & (dc["game_type"] == "REG") & (dc["week"] == 1)
    ].copy()

    dc_out = (
        dc_rb[["season", "gsis_id", "depth_team"]]
        .dropna(subset=["gsis_id"])
        .drop_duplicates(subset=["season", "gsis_id"], keep="first")
        .rename(columns={"gsis_id": "player_id", "depth_team": "depth_team_wk1"})
    )
    dc_out["depth_team_wk1"] = pd.to_numeric(dc_out["depth_team_wk1"], errors="coerce")
    log.info("Depth charts: %d player-season rows", len(dc_out))
    return dc_out

### Injury History
Weeks listed `Out`/`Doubtful` (`games_out_seas`) and `Questionable`/`Limited` (`games_limited_seas`) per season. YoY shift gives `games_out_seas_prev`.


In [ ]:
def load_injury_history(years: list[int]) -> pd.DataFrame:
    """
    Per season: weeks listed Out (game missed) and weeks listed Limited/Questionable.
    Deduplicated to one entry per (player, week) before aggregating.
    Returns one row per (player_id, season).
    """
    log.info("Loading injury reports %d-%d", years[0], years[-1])
    inj = nfl.import_injuries(years=years)

    inj_rb = inj[
        (inj["position"] == "RB") & (inj["game_type"] == "REG")
    ].drop_duplicates(subset=["season", "week", "gsis_id"]).copy()

    inj_agg = (
        inj_rb.groupby(["season", "gsis_id"])
        .agg(
            games_out_seas=(
                "report_status",
                lambda s: s.isin(["Out", "Doubtful"]).sum(),
            ),
            games_limited_seas=(
                "report_status",
                lambda s: s.isin(["Questionable", "Limited"]).sum(),
            ),
        )
        .reset_index()
        .rename(columns={"gsis_id": "player_id"})
    )
    log.info("Injuries: %d player-season rows", len(inj_agg))
    return inj_agg

### NGS Rushing Metrics (2016+)
Key skill metrics from NFL Next Gen Stats — available for RBs only:
- `ngs_ryoe_per_att` — rush yards over expected per attempt (pure skill, blocks out)
- `ngs_efficiency` — composite rushing efficiency score
- `ngs_pct_gte_8_defenders` — % of rushes into a loaded box (opportunity difficulty)
- `ngs_avg_time_to_los` — seconds from snap to line of scrimmage (explosiveness)


In [ ]:
def load_ngs_rushing(years: list[int]) -> pd.DataFrame:
    """
    NGS rushing metrics for RBs (2016+ only).
    Week-0 rows are season-level aggregates.

    Key features:
      ngs_efficiency              — yards per rush attempt above expectation
      ngs_pct_gte_8_defenders     — % of rushes vs 8+ men in box (opportunity difficulty)
      ngs_avg_time_to_los         — seconds from snap to crossing line of scrimmage
      ngs_ryoe_per_att            — rush yards over expected per attempt (pure skill)
      ngs_rush_pct_over_expected  — % of rushes exceeding expected yards
    """
    available = [y for y in years if y >= 2016]
    if not available:
        return pd.DataFrame(columns=["player_id", "season"])

    log.info("Loading NGS rushing %d-%d", available[0], available[-1])
    ngs = nfl.import_ngs_data(stat_type="rushing", years=available)
    ngs_rb = ngs[(ngs["week"] == 0) & (ngs["player_position"] == "RB")].copy()
    ngs_rb = ngs_rb.rename(columns={
        "player_gsis_id":                      "player_id",
        "efficiency":                          "ngs_efficiency",
        "percent_attempts_gte_eight_defenders": "ngs_pct_gte_8_defenders",
        "avg_time_to_los":                     "ngs_avg_time_to_los",
        "rush_yards_over_expected_per_att":    "ngs_ryoe_per_att",
        "rush_pct_over_expected":              "ngs_rush_pct_over_expected",
    })

    keep = [
        "player_id", "season",
        "ngs_efficiency", "ngs_pct_gte_8_defenders",
        "ngs_avg_time_to_los", "ngs_ryoe_per_att", "ngs_rush_pct_over_expected",
    ]
    log.info("NGS rushing: %d RB-season rows", len(ngs_rb))
    return ngs_rb[[c for c in keep if c in ngs_rb.columns]]


# ---------------------------------------------------------------------------
# Feature engineering
# ---------------------------------------------------------------------------

_PER_GAME_COLS = [
    "carries", "targets", "receptions",
    "rushing_yards", "receiving_yards",
    "receiving_air_yards", "receiving_yards_after_catch",
    "rushing_tds", "receiving_tds",
    "rushing_epa", "receiving_epa",
    "rushing_first_downs", "receiving_first_downs",
    "rushing_fumbles", "receiving_fumbles",
    "rushing_fumbles_lost", "receiving_fumbles_lost",
    "special_teams_tds",
    "gl_carries", "gl_targets", "gl_touches",
    "fantasy_points_ppr",
]

_YOY_BASE = [
    "ppg", "carries_pg", "targets_pg",
    "total_yards_pg", "total_epa_pg",
    "avg_snap_pct", "games_out_seas",
]

## Feature Engineering

`build_features()` merges all data sources and produces the full 61-feature set:
- Per-game versions of all volume / efficiency / scoring stats
- Year-over-year deltas for key metrics
- Derived ratios: `aDOT`, `carries_share`, `catch_rate`, `scrimmage_yds_per_touch`
- NGS rushing skill metrics (2016+ only; `NaN` for earlier seasons)


In [ ]:
_PER_GAME_COLS = [
    "carries", "targets", "receptions",
    "rushing_yards", "receiving_yards",
    "receiving_air_yards", "receiving_yards_after_catch",
    "rushing_tds", "receiving_tds",
    "rushing_epa", "receiving_epa",
    "rushing_first_downs", "receiving_first_downs",
    "rushing_fumbles", "receiving_fumbles",
    "rushing_fumbles_lost", "receiving_fumbles_lost",
    "special_teams_tds",
    "gl_carries", "gl_targets", "gl_touches",
    "fantasy_points_ppr",
]

_YOY_BASE = [
    "ppg", "carries_pg", "targets_pg",
    "total_yards_pg", "total_epa_pg",
    "avg_snap_pct", "games_out_seas",
]

In [ ]:
def build_features(
    seasonal:     pd.DataFrame,
    goal_line:    Optional[pd.DataFrame],
    team_carries: Optional[pd.DataFrame],
    ngs:          Optional[pd.DataFrame],
    snaps:        pd.DataFrame,
    depth_charts: pd.DataFrame,
    injuries:     pd.DataFrame,
) -> pd.DataFrame:
    """
    Merges all data sources and engineers the full feature set.
    Returns one row per (player_id, season) with games >= MIN_GAMES.
    """
    df = seasonal.copy()

    # --- goal-line touches
    if goal_line is not None and not goal_line.empty:
        df = pd.merge(df, goal_line, on=["player_id", "season"], how="left")
        for c in ("gl_carries", "gl_targets", "gl_touches"):
            df[c] = df[c].fillna(0).astype(int)
    else:
        for c in ("gl_carries", "gl_targets", "gl_touches"):
            df[c] = 0

    # --- team carry share (requires team carries from PBP)
    if team_carries is not None and not team_carries.empty:
        df = pd.merge(df, team_carries, on=["season", "team"], how="left")
        df["carries_share"] = np.where(
            df["team_carries_seas"] > 0,
            df["carries"] / df["team_carries_seas"],
            np.nan,
        )
    else:
        df["carries_share"] = np.nan

    # --- NGS rushing (left join; pre-2016 rows will be NaN — expected)
    if ngs is not None and not ngs.empty:
        df = pd.merge(df, ngs, on=["player_id", "season"], how="left")

    # --- snap counts
    if not snaps.empty:
        df = pd.merge(df, snaps, on=["player_id", "season"], how="left")

    # --- week-1 depth chart
    if not depth_charts.empty:
        df = pd.merge(df, depth_charts, on=["player_id", "season"], how="left")
        df["is_rb1_wk1"] = (df["depth_team_wk1"] == 1).astype(float)
        df.loc[df["depth_team_wk1"].isna(), "is_rb1_wk1"] = np.nan

    # --- injury history
    if not injuries.empty:
        df = pd.merge(df, injuries, on=["player_id", "season"], how="left")
        df["games_out_seas"]     = df["games_out_seas"].fillna(0)
        df["games_limited_seas"] = df["games_limited_seas"].fillna(0)

    # sleeper_id and birth_date already present from import_seasonal_rosters()

    # --- min games filter
    df = df[df["games"] >= MIN_GAMES].copy()

    # --- experience
    df["rookie_year"] = pd.to_numeric(df["rookie_year"], errors="coerce")
    df["years_exp"]   = (df["season"] - df["rookie_year"]).clip(lower=0)

    # --- per-game features
    for col in _PER_GAME_COLS:
        if col in df.columns:
            df[f"{col}_pg"] = df[col] / df["games"]

    # --- composite volume / efficiency
    df["touches"]        = df["carries"].fillna(0)   + df["receptions"].fillna(0)
    df["touches_pg"]     = df["touches"]    / df["games"]
    df["total_yards"]    = df["rushing_yards"].fillna(0) + df["receiving_yards"].fillna(0)
    df["total_yards_pg"] = df["total_yards"] / df["games"]
    df["total_epa"]      = df["rushing_epa"].fillna(0)  + df["receiving_epa"].fillna(0)
    df["total_epa_pg"]   = df["total_epa"]   / df["games"]
    df["total_tds"]      = df["rushing_tds"].fillna(0)  + df["receiving_tds"].fillna(0)
    df["total_tds_pg"]   = df["total_tds"]   / df["games"]

    # --- efficiency ratios (NaN when denominator is zero)
    df["ypc"]   = np.where(df["carries"]    > 0, df["rushing_yards"]   / df["carries"],    np.nan)
    df["ypr"]   = np.where(df["receptions"] > 0, df["receiving_yards"] / df["receptions"], np.nan)
    df["catch_rate"] = np.where(df["targets"] > 0, df["receptions"] / df["targets"], np.nan)
    df["scrimmage_yds_per_touch"] = np.where(
        df["touches"] > 0, df["total_yards"] / df["touches"], np.nan
    )
    # average depth of target: how deep in the air are balls thrown to this RB?
    df["aDOT"] = np.where(
        df["targets"] > 0, df["receiving_air_yards"] / df["targets"], np.nan
    )

    # --- cast demographics
    for col in ("height", "weight", "draft_number", "age"):
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # --- year-over-year deltas and lag features
    df = df.sort_values(["player_id", "season"]).reset_index(drop=True)
    for metric in _YOY_BASE:
        if metric in df.columns:
            df[f"{metric}_prev"] = df.groupby("player_id")[metric].shift(1)
            df[f"{metric}_yoy"]  = df[metric] - df[f"{metric}_prev"]

    # games_prev as a standalone feature (health/availability signal)
    df["games_prev"] = df.groupby("player_id")["games"].shift(1)

    return df


# ---------------------------------------------------------------------------
# Feature manifest
# ---------------------------------------------------------------------------

FEATURE_COLS: list[str] = [
    # volume (per game)
    "carries_pg", "targets_pg", "receptions_pg", "touches_pg",
    "rushing_yards_pg", "receiving_yards_pg",
    "receiving_air_yards_pg", "receiving_yards_after_catch_pg",
    "total_yards_pg",
    # scoring / EPA
    "rushing_tds_pg", "receiving_tds_pg", "total_tds_pg",
    "rushing_epa_pg", "receiving_epa_pg", "total_epa_pg",
    # first downs
    "rushing_first_downs_pg", "receiving_first_downs_pg",
    # goal line
    "gl_carries_pg", "gl_targets_pg", "gl_touches_pg",
    # efficiency
    "ypc", "ypr", "catch_rate", "scrimmage_yds_per_touch", "aDOT",
    # opportunity share (team context — already ratios)
    "tgt_sh", "ay_sh", "yac_sh", "ry_sh", "rtd_sh", "dom", "w8dom",
    # carries share (new: player / team total carries)
    "carries_share",
    # snap counts (new)
    "avg_snap_pct", "off_snaps_pg",
    # depth chart (new)
    "depth_team_wk1", "is_rb1_wk1",
    # ball security
    "rushing_fumbles_pg", "receiving_fumbles_pg",
    "rushing_fumbles_lost_pg", "receiving_fumbles_lost_pg",
    # injury history (new)
    "games_out_seas", "games_limited_seas",
    "games_out_seas_prev",       # prior season games missed
    "games_prev",                # prior season games played
    # NGS rushing (2016+ only; NaN before)
    "ngs_efficiency", "ngs_pct_gte_8_defenders",
    "ngs_avg_time_to_los", "ngs_ryoe_per_att", "ngs_rush_pct_over_expected",
    # year-over-year deltas
    "ppg_yoy", "carries_pg_yoy", "targets_pg_yoy",
    "total_yards_pg_yoy", "total_epa_pg_yoy",
    "avg_snap_pct_yoy",
    # demographics
    "age", "height", "weight", "draft_number", "years_exp",
]

ID_COLS: list[str] = [
    "player_id", "sleeper_id", "player_name",
    "team", "season", "games", "ppg",
    "college", "rookie_year", "birth_date",
]


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

## Feature Manifest

`FEATURE_COLS` is the exact list fed to the model matrix (61 features).  
`ID_COLS` are kept as reference columns but excluded from training.


In [ ]:
FEATURE_COLS: list[str] = [
    # volume (per game)
    "carries_pg", "targets_pg", "receptions_pg", "touches_pg",
    "rushing_yards_pg", "receiving_yards_pg",
    "receiving_air_yards_pg", "receiving_yards_after_catch_pg",
    "total_yards_pg",
    # scoring / EPA
    "rushing_tds_pg", "receiving_tds_pg", "total_tds_pg",
    "rushing_epa_pg", "receiving_epa_pg", "total_epa_pg",
    # first downs
    "rushing_first_downs_pg", "receiving_first_downs_pg",
    # goal line
    "gl_carries_pg", "gl_targets_pg", "gl_touches_pg",
    # efficiency
    "ypc", "ypr", "catch_rate", "scrimmage_yds_per_touch", "aDOT",
    # opportunity share (team context — already ratios)
    "tgt_sh", "ay_sh", "yac_sh", "ry_sh", "rtd_sh", "dom", "w8dom",
    # carries share (new: player / team total carries)
    "carries_share",
    # snap counts (new)
    "avg_snap_pct", "off_snaps_pg",
    # depth chart (new)
    "depth_team_wk1", "is_rb1_wk1",
    # ball security
    "rushing_fumbles_pg", "receiving_fumbles_pg",
    "rushing_fumbles_lost_pg", "receiving_fumbles_lost_pg",
    # injury history (new)
    "games_out_seas", "games_limited_seas",
    "games_out_seas_prev",       # prior season games missed
    "games_prev",                # prior season games played
    # NGS rushing (2016+ only; NaN before)
    "ngs_efficiency", "ngs_pct_gte_8_defenders",
    "ngs_avg_time_to_los", "ngs_ryoe_per_att", "ngs_rush_pct_over_expected",
    # year-over-year deltas
    "ppg_yoy", "carries_pg_yoy", "targets_pg_yoy",
    "total_yards_pg_yoy", "total_epa_pg_yoy",
    "avg_snap_pct_yoy",
    # demographics
    "age", "height", "weight", "draft_number", "years_exp",
]

ID_COLS: list[str] = [
    "player_id", "sleeper_id", "player_name",
    "team", "season", "games", "ppg",
    "college", "rookie_year", "birth_date",
]

---
## Step-by-Step Build

Run the cells below in order. Each step logs progress to the cell output.


### Step 1 — ID Crosswalk

In [ ]:
years = list(range(START_YEAR, END_YEAR + 1))
id_crosswalk = load_id_crosswalk()
id_crosswalk.head(3)

### Step 2 — Seasonal Stats + Rosters

In [ ]:
seasonal = load_seasonal_rb(years)
print(f'{len(seasonal)} RB-season rows | {seasonal["season"].nunique()} seasons')
seasonal[['player_name', 'season', 'team', 'games', 'ppg',
          'height', 'weight', 'draft_number', 'sleeper_id']].head(5)

### Step 3 — Snap Counts

In [ ]:
snaps = load_snap_counts(years, id_crosswalk)
print(f'{len(snaps)} player-season rows')
snaps.head(5)

### Step 4 — Depth Charts

In [ ]:
depth_charts = load_depth_charts(years)
print(f'{len(depth_charts)} player-season rows')
depth_charts.head(5)

### Step 5 — Injury History

In [ ]:
injuries = load_injury_history(years)
print(f'{len(injuries)} player-season rows')
injuries[injuries['games_out_seas'] > 0].head(5)

### Step 6 — NGS Rushing

In [ ]:
ngs = load_ngs_rushing(years)
print(f'{len(ngs)} RB-season rows')
ngs.head(5)

### Step 7 — PBP: Goal-Line Touches + Team Carry Totals

Controlled by `SKIP_PBP` in Config.  
⚠️ Allow 20–40 minutes when `SKIP_PBP = False`.


In [ ]:
if SKIP_PBP:
    goal_line, team_carries = None, None
    print('PBP skipped — gl_*_pg and carries_share will be NaN')
else:
    goal_line, team_carries = load_pbp_features(years)
    print(f'Goal-line rows: {len(goal_line)} | Team-carry rows: {len(team_carries)}')

### Step 8 — Build Feature Matrix

In [ ]:
features = build_features(
    seasonal, goal_line, team_carries,
    ngs, snaps, depth_charts, injuries,
)
print(f'Shape: {features.shape}')
features.head(3)

### Step 9 — Audit & Save

In [ ]:
present = [c for c in FEATURE_COLS if c in features.columns]
missing = [c for c in FEATURE_COLS if c not in features.columns]
print(f'Features present : {len(present)} / {len(FEATURE_COLS)}')
if missing:
    print(f'Missing          : {missing}')

out_cols = ID_COLS + [c for c in FEATURE_COLS if c in features.columns]
out_df   = features[[c for c in out_cols if c in features.columns]].copy()

try:
    out_df.to_parquet(OUT_PARQUET, engine='fastparquet', index=False)
    print(f'Saved -> {OUT_PARQUET}')
except Exception as e:
    print(f'Parquet failed ({e})')

out_df.to_csv(OUT_CSV, index=False)
print(f'Saved -> {OUT_CSV}')
print(f'\n{len(out_df)} rows | {out_df["player_id"].nunique()} unique RBs | {out_df["season"].nunique()} seasons')

---
## Explore Output

Load the saved CSV and inspect the dataset.

### Dataset Overview

In [ ]:
import pandas as pd
pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)
pd.set_option('display.float_format', '{:.3f}'.format)

df = pd.read_csv(OUT_CSV)
print(f'Shape  : {df.shape}')
print(f'Seasons: {df["season"].min()} - {df["season"].max()}')
print(f'Sleeper ID match : {df["sleeper_id"].notna().mean():.1%}')
print(f'NGS coverage     : {df["ngs_ryoe_per_att"].notna().sum()} rows')
print(f'Snap coverage    : {df["avg_snap_pct"].notna().sum()} rows')
print(f'Injury rows      : {(df["games_out_seas"] > 0).sum()} player-seasons with >=1 game Out')
df.describe()

### Top PPG Seasons — New Features

In [ ]:
show = [
    'player_name', 'season', 'years_exp',
    'carries_share', 'avg_snap_pct', 'depth_team_wk1', 'is_rb1_wk1',
    'aDOT', 'games_out_seas', 'games_prev',
    'ngs_ryoe_per_att', 'ngs_pct_gte_8_defenders',
    'carries_pg_yoy', 'ppg_yoy', 'ppg',
]
df.nlargest(15, 'ppg')[show]

### Injury-Impacted Seasons

In [ ]:
inj = df[df['games_out_seas'] > 2].sort_values('ppg', ascending=False)
inj[['player_name', 'season', 'games', 'games_prev', 'games_out_seas',
     'games_limited_seas', 'avg_snap_pct', 'carries_pg', 'ppg']].head(12)

### Rookie Breakout Candidates (years_exp = 0, PPG > 10)

In [ ]:
rooks = df[(df['years_exp'] == 0) & (df['ppg'] > 10)].sort_values('ppg', ascending=False)
rooks[['player_name', 'season', 'draft_number', 'depth_team_wk1',
       'carries_pg', 'targets_pg', 'avg_snap_pct', 'aDOT',
       'ngs_ryoe_per_att', 'ppg']].head(12)

### Sleeper Trending Adds (Live API Demo)

Fetches the top trending waiver-wire adds in the past 48 hours and cross-references them against the most recent season in your dataset.


In [ ]:
client  = SleeperClient()
trending = client.fetch_trending(trend_type='add', limit=25)

latest_season = df['season'].max()
recent = df[df['season'] == latest_season][['sleeper_id', 'player_name', 'ppg',
                                              'carries_pg', 'avg_snap_pct']].copy()
recent['sleeper_id'] = recent['sleeper_id'].astype(str)

if 'player_id' in trending.columns:
    merged = trending.merge(recent, left_on='player_id', right_on='sleeper_id', how='left')
    print(merged[['player_id', 'count', 'player_name', 'ppg', 'carries_pg', 'avg_snap_pct']]
          .sort_values('count', ascending=False)
          .to_string(index=False))
else:
    print(trending)